In [ ]:
!pip install --upgrade pip -q
!pip install ultralytics roboflow huggingface_hub -q

import gc
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Executing on device: {torch.cuda.get_device_name(0)}")
    print(f"Total vRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 25.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2026.2.0 which is incompatible.
CUDA Available: True
Executing on device: Tesla T4
Total vRAM: 15.64 GB


In [2]:
import os
import shutil
from roboflow import Roboflow
from ultralytics import YOLO

# Define absolute output pathways
WORKING_DIR = "/kaggle/working"
EXPORT_WEIGHTS_DIR = os.path.join(WORKING_DIR, "production_sports_weights")

# Create persistent output directory
os.makedirs(EXPORT_WEIGHTS_DIR, exist_ok=True)
print(f"Workspace established. Final ONNX models will be saved to: {EXPORT_WEIGHTS_DIR}")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Workspace established. Final ONNX models will be saved to: /kaggle/working/production_sports_weights


In [ ]:
from kaggle_secrets import UserSecretsClient

# Fetch API token securely
try:
    user_secrets = UserSecretsClient()
    ROBOFLOW_API_KEY = user_secrets.get_secret("ROBOFLOW_KEY")
except Exception:
    ROBOFLOW_API_KEY = "YOUR_API_KEY_HERE"

rf = Roboflow(api_key=ROBOFLOW_API_KEY)

print("Ingesting Football Scouting Segmentation Asset...")
football_project = rf.workspace("school-zlexb").project("football-scouting")
football_dataset = football_project.version(5).download("yolov11")

print("\nIngesting Formula 1 Bounding Box Detection Asset...")
f1_project = rf.workspace("alens-workspace-wdlhr").project("f1-detection-2025")
f1_dataset = f1_project.version(6).download("yolov11")

print("\nData successfully localized!")

Ingesting Football Scouting Segmentation Asset...
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Football-Scouting-5 in yolov11:: 100%|██████████| 3395/3395 [00:00<00:00, 4115.91it/s]

loading Roboflow workspace...


loading Roboflow project...



Extracting Dataset Version Zip to F1-detection-2025-6 in yolov11:: 100%|██████████| 7953/7953 [00:00<00:00, 9374.35it/s] 


Data successfully localized!


In [4]:
print("Loading Pre-trained YOLOv11 EXTRA-LARGE Segmentation Weights (Football)...")
# Using .pt instead of .yaml starts training from advanced COCO weights
football_model = YOLO('yolo11x-seg.pt') 

print("\nLoading Pre-trained YOLOv11 LARGE Detection Weights (F1)...")
# Large variant provides blazing speed for bounding boxes without sacrificing accuracy
f1_model = YOLO('yolo11l.pt')

print("\nFoundation models loaded into GPU memory!")

Loading Pre-trained YOLOv11 EXTRA-LARGE Segmentation Weights (Football)...

Loading Pre-trained YOLOv11 LARGE Detection Weights (F1)...

Foundation models loaded into GPU memory!


In [5]:
import os
import gc
import torch
from ultralytics import YOLO

# MLOPS SAFEGUARD: Prevents PyTorch from crashing due to memory fragmentation
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

# Purge any remaining tensor artifacts from previous execution crashes
gc.collect()
torch.cuda.empty_cache()

# 1. FOOTBALL SEGMENTATION CONFIGURATION (640px Optimized)
football_hyperparameters = {
    'epochs': 60,
    'imgsz': 640,            # Downscaled to 640 to save massive amounts of vRAM
    'batch': 8,              # Safely scaled up to 8 now that image size is smaller
    'device': 0,
    'optimizer': 'AdamW',
    'lr0': 0.001,
    'lrf': 0.01,
    'warmup_epochs': 5,
    'cache': True,           # Caches images in RAM to accelerate epoch computation speeds
    'workers': 4,
    'amp': True,             # Mixed precision execution cuts memory footprint by ~40%
    'mask_ratio': 4,         # Keeps internal pooling stable to eliminate mask BCE errors
    'overlap_mask': True,    # Handles overlapping player segmentation boundaries
    
    # Advanced Data Augmentations
    'mosaic': 1.0,
    'mixup': 0.1,
    'degrees': 15.0,
    'scale': 0.6,
    'perspective': 0.0003,
    'hsv_v': 0.4,
    'hsv_s': 0.7,
    'fliplr': 0.5,
    'close_mosaic': 10,
    
    # Callback Performance Settings
    'val': True,
    'save': True,
    'save_period': 10,
    'patience': 20,
    'exist_ok': True,
    'plots': True
}

print("Launching Memory-Optimized Football Segmentation Core Run...")
football_model.train(
    data=f"{football_dataset.location}/data.yaml",
    project=f"{WORKING_DIR}/training_runs",
    name="football_seg_run",
    **football_hyperparameters
)


# Explicit cleanup boundary to isolate the two sports executions completely
gc.collect()
torch.cuda.empty_cache()


# 2. FORMULA 1 DETECTION CONFIGURATION (640px Optimized)
f1_hyperparameters = {
    'epochs': 50,
    'imgsz': 640,            # Synchronized resolution footprint 
    'batch': 16,             # Bounding box detection handles large batches easily at 640px
    'device': 0,
    'optimizer': 'AdamW',
    'lr0': 0.01,
    'warmup_epochs': 3,
    'cache': True,
    'workers': 4,
    'amp': True,
    'cls': 0.3,              # Mitigates team livery color distribution bias
    
    # Advanced Data Augmentations
    'mosaic': 0.8,
    'mixup': 0.05,
    'degrees': 5.0,
    'scale': 0.75,
    'perspective': 0.0002,
    'hsv_v': 0.3,
    'hsv_s': 0.5,
    'fliplr': 0.5,
    'close_mosaic': 10,
    'copy_paste': 0.3,       # Pastes rare vehicle classes onto new tracks
    
    # Callback Performance Settings
    'val': True,
    'save': True,
    'save_period': 10,
    'patience': 20,
    'exist_ok': True,
    'plots': True
}

print("\nLaunching Memory-Optimized F1 Detection Core Run...")
f1_model.train(
    data=f"{f1_dataset.location}/data.yaml",
    project=f"{WORKING_DIR}/training_runs",
    name="f1_det_run",
    **f1_hyperparameters
)

Launching Memory-Optimized Football Segmentation Core Run...
Ultralytics 8.4.51 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/Football-Scouting-5/data.yaml, degrees=15.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=60, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolo11x-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=football_seg_run, nbs=64,

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7bd2795f2720>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.0

In [6]:
print("Initializing Production-Grade ONNX Export Sequence...")

# Define structural pathways back to the best performing checkpoints
best_football_pt = f"{WORKING_DIR}/training_runs/football_seg_run/weights/best.pt"
best_f1_pt = f"{WORKING_DIR}/training_runs/f1_det_run/weights/best.pt"

# Pull best converged parameters back into tracking graphs
final_football_model = YOLO(best_football_pt)
final_f1_model = YOLO(best_f1_pt)

# Export models to ONNX using Claude's strict deployment rules
# dynamic=True ensures the web UI backend can accept larger frame sizes at runtime
print("Compiling Football Core to Dynamic ONNX...")
final_football_model.export(
    format='onnx',
    imgsz=640,
    dynamic=True,
    half=False             # Keeps weights at FP32 for universal CPU server deployment compatibility
)

print("Compiling Formula 1 Core to Dynamic ONNX...")
final_f1_model.export(
    format='onnx',
    imgsz=640,
    dynamic=True,
    half=False
)

# Relocate compilation assets into your clean distribution folder
shutil.copy(f"{WORKING_DIR}/training_runs/football_seg_run/weights/best.onnx", 
            os.path.join(EXPORT_WEIGHTS_DIR, "football_production_core.onnx"))
shutil.copy(f"{WORKING_DIR}/training_runs/f1_det_run/weights/best.onnx", 
            os.path.join(EXPORT_WEIGHTS_DIR, "f1_production_core.onnx"))

print("\nVerification Check: Active Production Storage Assets:")
for file in os.listdir(EXPORT_WEIGHTS_DIR):
    file_path = os.path.join(EXPORT_WEIGHTS_DIR, file)
    size_mb = os.path.getsize(file_path) / (1024 * 1024)
    print(f" -> Production Core: {file} | Size: {size_mb:.2f} MB")

print("\nVerification complete! The pipeline is completely secure. Fire up the cells!")

Initializing Production-Grade ONNX Export Sequence...
Compiling Football Core to Dynamic ONNX...
Ultralytics 8.4.51 🚀 Python-3.12.12 torch-2.10.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
YOLO11x-seg summary (fused): 204 layers, 62,006,748 parameters, 0 gradients, 295.9 GFLOPs

PyTorch: starting from '/kaggle/working/training_runs/football_seg_run/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) ((1, 40, 8400), (1, 32, 160, 160)) (119.0 MB)
requirements: Ultralytics requirements ['onnxslim>=0.1.71', 'onnxruntime-gpu'] not found, attempting AutoUpdate...
Using Python 3.12.12 environment at: /usr
Resolved 12 packages in 262ms
 Downloaded onnxruntime-gpu
Prepared 2 packages in 2.89s
Installed 2 packages in 12ms
 + onnxruntime-gpu==1.26.0
 + onnxslim==0.1.93

requirements: AutoUpdate success ✅ 3.7s
WARNING ⚠️ requirements: Resta